In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import yaml

%matplotlib tk

In [ ]:
energy_df = pd.read_excel('Dataset.xlsx', sheet_name='2023 data', usecols='A:F', nrows=8762)
meta_data = yaml.safe_load(open("meta_data.yml", "r"))
energy_df.tail()

In [ ]:
energy_df.info()

In [ ]:
# plot all the columns
energy_df.plot(subplots=True, figsize=(10, 12))
plt.tight_layout()

## Daily data

In [ ]:
date = "2023-01-08"
daily_data = energy_df[energy_df['Date'].dt.date == pd.to_datetime(date).date()]
daily_data.plot(subplots=True, figsize=(10, 12))
plt.tight_layout()

In [ ]:
print(energy_df['Date'].dt.date[0] == pd.to_datetime(date).date())
print(energy_df['Date'].dt.date[0], date)

## Seasonal data

In [ ]:
seaons = meta_data['seasons']
season_dict = {seas: pd.read_excel('Dataset.xlsx', sheet_name=seas) for seas in seaons}

In [ ]:
# Season-wise consumption data

# build long-form data for violin plot
consumption_long = pd.concat(
    [
        season_dict[seas][['Consumption (kW)']].assign(Season=seas)
        for seas in seaons
    ],
    ignore_index=True
 )

# violin plot of consumption by season
plt.figure(figsize=(10, 6))
ax = sns.violinplot(
    data=consumption_long,
    x='Season',
    y='Consumption (kW)',
    inner='quartile',
    cut=0
 )
ax.set_title('Season-wise Consumption (kW) Distribution')
ax.set_xlabel('Season')
ax.set_ylabel('Consumption (kW)')
plt.tight_layout()

In [ ]:
# Season-wise price data

# build long-form data for violin plot
price_long = pd.concat(
    [
        season_dict[seas][['Energy price (EUR/kWh)']].assign(Season=seas)
        for seas in seaons
    ],
    ignore_index=True
 )

# violin plot of consumption by season
plt.figure(figsize=(10, 6))
ax = sns.violinplot(
    data=price_long,
    x='Season',
    y='Energy price (EUR/kWh)',
    inner='quartile',
    cut=0,
    color='lightcoral'
 )
ax.set_title('Season-wise Energy Price (EUR/kWh) Distribution')
ax.set_xlabel('Season')
ax.set_ylabel('Energy Price (EUR/kWh)')
plt.tight_layout()

- The energy prices high variance in winter and autumn seasons compared to summer. Supply contraints and demand surge during winter and autumn seasons could be the reason for this high variance. In summer, the demand is relatively stable, leading to lower price volatility.

## Scenario data

In [ ]:
season = 'Winter'
n_scenes = 3
sheet_name = f"Set of {n_scenes} {season.lower()} scenarios"
scenarios_df = pd.read_excel('Dataset.xlsx', sheet_name=sheet_name)

scenarios_df.head()

In [ ]:
# Check probabilities for all scenarios
num_scenerios = meta_data['num_scenarios']

for season in seaons:
    for n in num_scenerios:
        sheet_name = f"Set of {n} {season.lower()} scenarios"
        scenarios_df = pd.read_excel('Dataset.xlsx', sheet_name=sheet_name)
        probs = scenarios_df['Probability (%)'].values
        probs = probs[::24]
        print(f"{season} - {len(probs)}/{n} - {np.sum(probs)}")
        assert np.isclose(np.sum(probs), 1.0, rtol=1e-3), f"Probabilities for {season} with {n} scenarios do not sum to 1!"
